In [18]:
import pandas as pd
import numpy as np

In [20]:
import pandas as pd
import numpy as np

# ---------- 1) Build EWJ daily from hourly ----------
ewj = pd.read_csv("./EWJ.csv")
ewj["date"] = pd.to_datetime(ewj["date"])
ewj = ewj.sort_values("date")

ewj_daily = (
    ewj.set_index("date")
    .resample("D")
    .agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    .dropna(subset=["open", "high", "low", "close"])
)

# ---------- 2) Load NKY dates (master calendar) ----------
nky = pd.read_csv("./NKY.csv")
nky["date"] = pd.to_datetime(nky["date"])
nky = nky.sort_values("date")
nky_dates = pd.DatetimeIndex(nky["date"]).unique()

# ---------- 3) Align EWJ to NKY calendar ----------
# Keep actual EWJ close where available, NaN otherwise
close_actual = ewj_daily["close"].reindex(nky_dates)

if close_actual.notna().sum() == 0:
    raise ValueError("No overlapping dates between EWJ and NKY calendars.")

first_actual_date = close_actual.first_valid_index()
first_actual_close = float(close_actual.loc[first_actual_date])

# ---------- 4) Splice pre-EWJ history using NKY return path ----------
nky_close = nky.set_index("date")["close"].reindex(nky_dates)
nky_ret = nky_close.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)

close_spliced = close_actual.copy()
pre_dates = nky_dates[nky_dates < first_actual_date]

if len(pre_dates) > 0:
    # Backward recursion: P_t = P_{t+1} / (1 + r_{t+1}^{NKY})
    prev_price = first_actual_close
    for d in pre_dates[::-1]:
        next_d = nky_dates[nky_dates.get_loc(d) + 1]
        r_next = float(nky_ret.loc[next_d])
        denom = 1.0 + r_next
        if abs(denom) < 1e-12:
            denom = 1e-12
        prev_price = prev_price / denom
        close_spliced.loc[d] = prev_price

# For post-start non-overlap holidays, carry previous EWJ close
close_spliced = close_spliced.ffill()

# ---------- 5) Build final daily dataframe ----------
ewj_daily_aligned = pd.DataFrame(index=nky_dates)
ewj_daily_aligned["close"] = close_spliced

# Use actual OHLCV where available; synthetic rows get OHLC=close, volume=0
for col in ["open", "high", "low", "volume"]:
    ewj_daily_aligned[col] = ewj_daily[col].reindex(nky_dates)

synthetic_mask = ewj_daily_aligned[["open", "high", "low"]].isna().any(axis=1)
ewj_daily_aligned.loc[synthetic_mask, "open"] = ewj_daily_aligned.loc[synthetic_mask, "close"]
ewj_daily_aligned.loc[synthetic_mask, "high"] = ewj_daily_aligned.loc[synthetic_mask, "close"]
ewj_daily_aligned.loc[synthetic_mask, "low"] = ewj_daily_aligned.loc[synthetic_mask, "close"]
ewj_daily_aligned["volume"] = ewj_daily_aligned["volume"].fillna(0.0)

ewj_daily_aligned["source"] = "actual_EWJ"
ewj_daily_aligned.loc[synthetic_mask, "source"] = "synthetic_from_NKY"

ewj_daily_aligned.index.name = "date"
ewj_daily_aligned = ewj_daily_aligned.reset_index()

# Same order as benchmark-style files
ewj_daily_aligned = ewj_daily_aligned[["date", "close", "high", "low", "open", "volume", "source"]]

print("first_actual_date:", first_actual_date)
print("rows:", len(ewj_daily_aligned))
print(ewj_daily_aligned.head())
print(ewj_daily_aligned.tail())

first_actual_date: 2004-01-23 00:00:00
rows: 5676
        date      close       high        low       open  volume  \
0 2003-01-06  30.763090  30.763090  30.763090  30.763090     0.0   
1 2003-01-07  30.562447  30.562447  30.562447  30.562447     0.0   
2 2003-01-08  30.072755  30.072755  30.072755  30.072755     0.0   
3 2003-01-09  30.002602  30.002602  30.002602  30.002602     0.0   
4 2003-01-10  29.905583  29.905583  29.905583  29.905583     0.0   

               source  
0  synthetic_from_NKY  
1  synthetic_from_NKY  
2  synthetic_from_NKY  
3  synthetic_from_NKY  
4  synthetic_from_NKY  
           date  close   high    low   open      volume      source
5671 2026-03-09  85.65  85.87  82.91  83.78  17952984.0  actual_EWJ
5672 2026-03-10  86.46  88.14  85.86  86.50  11447023.0  actual_EWJ
5673 2026-03-11  85.75  86.18  84.98  85.30   6921082.0  actual_EWJ
5674 2026-03-12  84.18  84.92  83.76  84.82  11500709.0  actual_EWJ
5675 2026-03-13  83.36  84.79  83.21  84.53   8667997.0  

In [21]:
ewj_daily_aligned.to_csv("EWJ_NK.csv", index=False)